# Quantization Deep Dive

Quantization is *the* key technique in this challenge. It determines how much model you can fit in 16MB.

## Why Quantization Matters Here

| Format | Bytes per param | 3.3M params | Budget used |
|--------|----------------|-------------|-------------|
| float32 | 4 | 13.2 MB | 82.5% |
| float16 | 2 | 6.6 MB | 41.3% |
| int8 | 1 | 3.3 MB | 20.6% |
| int6 | 0.75 | 2.5 MB | 15.4% |
| int5 | 0.625 | 2.1 MB | 12.8% |

Going from int8 to int6 saves ~25% space — enough to add an entire extra layer or make the MLP 50% bigger!

In [ ]:
import sys
sys.path.insert(0, '../parameter-golf')

import numpy as np
import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten
import zlib
import pickle

from train_gpt_mlx import (
    Hyperparameters, GPT, quantize_state_dict_int8, dequantize_state_dict_int8,
    quantize_float_array, INT8_CLIP_PERCENTILE
)

## 1. How Int8 Quantization Works

The idea is simple: map floating-point values to integers in [-127, 127].

```
float_value ≈ int8_value × scale
```

For 2D matrices (attention/MLP weights), we use **per-row scaling**:
- Each row gets its own scale factor
- This captures different ranges across output channels
- Cost: one float16 scale per row (tiny overhead)

For vectors/scalars, we use **per-tensor scaling**:
- One scale factor for the whole tensor

In [ ]:
# Let's quantize a single weight matrix and see what happens
mx.random.seed(42)
# Simulate a trained weight matrix with realistic distribution
test_weight = mx.random.normal((512, 256)) * 0.02  # typical init scale

# Quantize it
q, scale = quantize_float_array(test_weight)

print("=== Int8 Quantization of a 512x256 matrix ===")
print(f"Original: dtype={test_weight.dtype}, range=[{float(mx.min(test_weight)):.4f}, {float(mx.max(test_weight)):.4f}]")
print(f"Quantized: dtype={q.dtype}, range=[{q.min()}, {q.max()}]")
print(f"Scales: dtype={scale.dtype}, shape={scale.shape}, range=[{scale.min():.6f}, {scale.max():.6f}]")
print(f"\nOriginal size:  {test_weight.nbytes:,} bytes")
print(f"Quantized size: {q.nbytes + scale.nbytes:,} bytes")
print(f"Compression:    {test_weight.nbytes / (q.nbytes + scale.nbytes):.2f}x")

# Dequantize and measure error
dequant = q.astype(np.float32) * scale[:, None]
orig = np.array(test_weight.astype(mx.float32))
error = np.abs(orig - dequant)
print(f"\nMax absolute error: {error.max():.6f}")
print(f"Mean absolute error: {error.mean():.6f}")
print(f"Relative error (mean): {(error / (np.abs(orig) + 1e-8)).mean():.4f}")

## 2. Int6 Quantization

The top submissions go further — quantizing to 6 bits (range [-32, 31]).

This is 25% smaller than int8 but introduces more quantization error. The trick is **Quantization-Aware Training (QAT)** — training the model while simulating quantization noise.

In [ ]:
def quantize_int6(arr_np):
    """Quantize a numpy float32 array to int6 [-32, 31] with per-row scaling."""
    if arr_np.ndim == 2:
        abs_max = np.max(np.abs(arr_np), axis=1)
        scale = np.maximum(abs_max / 31.0, 1.0 / 31.0).astype(np.float16)
        q = np.clip(np.round(arr_np / scale[:, None].astype(np.float32)), -32, 31).astype(np.int8)
        return q, scale
    else:
        abs_max = float(np.max(np.abs(arr_np)))
        scale = np.float16(abs_max / 31.0 if abs_max > 0 else 1.0)
        q = np.clip(np.round(arr_np / float(scale)), -32, 31).astype(np.int8)
        return q, scale

# Compare int8 vs int6
test_np = np.array(test_weight.astype(mx.float32))

q8, s8 = quantize_float_array(test_weight)
q6, s6 = quantize_int6(test_np)

# Dequantize both
deq8 = q8.astype(np.float32) * s8.astype(np.float32)[:, None]
deq6 = q6.astype(np.float32) * s6.astype(np.float32)[:, None]

err8 = np.abs(test_np - deq8)
err6 = np.abs(test_np - deq6)

print("=== Int8 vs Int6 Comparison ===")
print(f"{'':20s} {'Int8':>12s} {'Int6':>12s}")
print(f"{'Range':20s} {'[-127, 127]':>12s} {'[-32, 31]':>12s}")
print(f"{'Bytes (data)':20s} {q8.nbytes:>12,} {q6.nbytes:>12,}")
print(f"{'Bytes (scales)':20s} {s8.nbytes:>12,} {s6.nbytes:>12,}")
print(f"{'Total bytes':20s} {q8.nbytes + s8.nbytes:>12,} {q6.nbytes + s6.nbytes:>12,}")
print(f"{'Mean abs error':20s} {err8.mean():>12.6f} {err6.mean():>12.6f}")
print(f"{'Max abs error':20s} {err8.max():>12.6f} {err6.max():>12.6f}")
print(f"\nInt6 saves {q8.nbytes - q6.nbytes:,} bytes (wait — same bytes because both stored as int8!)")
print(f"The real savings come from compression: int6 values cluster in [-32,31] → compress better.")
print(f"Also, int6 has fewer unique values → better entropy → smaller after zlib/zstd.")

## 3. Compression: zlib vs zstd

After quantization, the weights are compressed. Let's compare the two main compressors.

In [ ]:
try:
    import zstandard as zstd
    has_zstd = True
except ImportError:
    has_zstd = False
    print("zstandard not installed, skipping zstd comparison")

# Compress the int8 quantized data
raw_bytes = q8.tobytes() + s8.tobytes()
zlib_compressed = zlib.compress(raw_bytes, level=9)

print(f"Raw quantized data: {len(raw_bytes):,} bytes")
print(f"After zlib-9:       {len(zlib_compressed):,} bytes ({len(zlib_compressed)/len(raw_bytes)*100:.1f}%)")

if has_zstd:
    compressor = zstd.ZstdCompressor(level=22)
    zstd_compressed = compressor.compress(raw_bytes)
    print(f"After zstd-22:      {len(zstd_compressed):,} bytes ({len(zstd_compressed)/len(raw_bytes)*100:.1f}%)")
    print(f"\nzstd saves {len(zlib_compressed) - len(zstd_compressed):,} bytes vs zlib ({(1 - len(zstd_compressed)/len(zlib_compressed))*100:.1f}% smaller)")

# Now compare with int6 data
raw_bytes_6 = q6.tobytes() + s6.tobytes()
zlib_6 = zlib.compress(raw_bytes_6, level=9)
print(f"\n--- Int6 compression ---")
print(f"Raw int6 data:  {len(raw_bytes_6):,} bytes")
print(f"After zlib-9:   {len(zlib_6):,} bytes ({len(zlib_6)/len(raw_bytes_6)*100:.1f}%)")
if has_zstd:
    zstd_6 = compressor.compress(raw_bytes_6)
    print(f"After zstd-22:  {len(zstd_6):,} bytes ({len(zstd_6)/len(raw_bytes_6)*100:.1f}%)")
    print(f"\nInt6+zstd vs Int8+zlib: saves {len(zlib_compressed) - len(zstd_6):,} bytes per matrix this size!")

## 4. Quantization-Aware Training (QAT)

The problem with post-training quantization: the model was never trained to handle quantization error.

**QAT solution:** During training, simulate quantization on the forward pass using the **Straight-Through Estimator (STE)**:

```python
# Forward: quantize and dequantize (adds noise)
w_fake_quant = dequantize(quantize(w))  

# Backward: pretend the quantization didn't happen
# Gradients flow through as if w_fake_quant == w
```

This teaches the model to be robust to quantization noise. The weights naturally gravitate toward values that quantize cleanly.

In [ ]:
def fake_quantize_int6(w):
    """Simulate int6 quantization during training (STE).
    Forward: quantize then dequantize (adds noise).
    Backward: straight-through (gradients pass unchanged).
    """
    w_np = np.array(w.astype(mx.float32))
    if w_np.ndim == 2:
        abs_max = np.max(np.abs(w_np), axis=1, keepdims=True)
        scale = np.maximum(abs_max / 31.0, 1.0 / 31.0)
        q = np.clip(np.round(w_np / scale), -32, 31)
        return mx.array((q * scale).astype(np.float32))
    else:
        abs_max = float(np.max(np.abs(w_np)))
        scale = max(abs_max / 31.0, 1.0 / 31.0)
        q = np.clip(np.round(w_np / scale), -32, 31)
        return mx.array((q * scale).astype(np.float32))

# Show the effect
w = mx.random.normal((8, 8)) * 0.1
w_fq = fake_quantize_int6(w)

print("Original weights:")
print(np.array(w.astype(mx.float32))[:4, :4].round(4))
print("\nAfter fake int6 quantization:")
print(np.array(w_fq.astype(mx.float32))[:4, :4].round(4))
print("\nDifference (quantization noise):")
diff = np.array((w - w_fq).astype(mx.float32))
print(diff[:4, :4].round(6))
print(f"\nNote: values are 'snapped' to the nearest int6 grid point.")
print(f"With QAT, the model learns to place weights ON these grid points.")

## 5. Full Model Size Analysis

Let's build the full baseline model and see exactly where the bytes go.

In [ ]:
# Build baseline model
args = Hyperparameters()
mx.random.seed(1337)
model = GPT(
    vocab_size=args.vocab_size, num_layers=args.num_layers, dim=args.model_dim,
    num_heads=args.num_heads, num_kv_heads=args.num_kv_heads, mlp_mult=args.mlp_mult,
    logit_chunk_tokens=0, logit_softcap=args.logit_softcap, rope_base=args.rope_base,
    tied_embed_init_std=args.tied_embed_init_std, qk_gain_init=args.qk_gain_init,
)

flat_state = {k: v for k, v in tree_flatten(model.state)}

# Categorize tensors
categories = {'Attention Q/K/V/Proj': 0, 'MLP fc/proj': 0, 'Embedding': 0, 'Scales/Controls': 0, 'Skip weights': 0}
for name, arr in flat_state.items():
    nbytes = int(arr.size) * (1 if arr.size > 65536 else 2)  # int8 for big, fp16 for small
    if 'c_q' in name or 'c_k' in name or 'c_v' in name or (name.endswith('proj.weight') and 'attn' in name):
        categories['Attention Q/K/V/Proj'] += nbytes
    elif 'mlp' in name and 'weight' in name:
        categories['MLP fc/proj'] += nbytes
    elif 'tok_emb' in name:
        categories['Embedding'] += nbytes
    elif 'skip_weight' in name:
        categories['Skip weights'] += nbytes
    else:
        categories['Scales/Controls'] += nbytes

total = sum(categories.values())
print("=== Approximate Byte Budget (int8 quantized, before compression) ===")
for cat, nbytes in sorted(categories.items(), key=lambda x: -x[1]):
    bar = '#' * int(nbytes / total * 50)
    print(f"{cat:25s} {nbytes:>10,} bytes ({nbytes/total*100:5.1f}%) {bar}")
print(f"{'Total':25s} {total:>10,} bytes")
print(f"\nMLP weights dominate! That's why 3x MLP expansion is such a big deal.")
print(f"Going int8→int6 on MLP alone saves ~{int(categories['MLP fc/proj'] * 0.25):,} bytes.")

## Key Takeaways

1. **Quantization is the budget lever** — int8→int6 frees ~25% space for more params
2. **Per-row scaling** is much better than per-tensor for weight matrices
3. **Compression amplifies savings** — int6 values have less entropy → compress smaller
4. **zstd > zlib** by ~5%, which matters at the margin
5. **QAT (STE)** trains the model to be quantization-friendly → less accuracy loss
6. **MLP weights are the biggest target** — optimizing them has the highest impact

Next: Let's look at the Muon optimizer and how it differs from Adam.